# Joint longitudinal–survival modeling — the current-value link

Every survival link Onkos has shipped is **two-stage**: collapse the tumor trajectory to one
static covariate (week-8 change, k_g, integrated burden), then apply a parametric/Cox baseline.
A static covariate means a **proportional hazard** — the hazard ratio between two tumors is
constant over time.

The joint longitudinal–survival model relaxes exactly that. The **current-value** link makes the
instantaneous hazard track the *current* tumor size:

$$\lambda(t) = \lambda_0(t)\,\exp\big(\alpha\,\log(v(t)/y_0)\big),\qquad S(t)=\exp\!\Big(-\!\int_0^t\lambda(u)\,du\Big)$$

*Population/trial level only; α and the baseline are declared, illustrative — never fitted here.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.joint import current_value_survival, joint_survival, compare_joint_vs_two_stage

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0.0, 260.0, 521)
SHAPE, SCALE = 1.3, 60.0   # the NSCLC week-8 Weibull baseline

## A strict generalization of proportional hazards

The cumulative hazard is integrated against the **analytic** baseline cumulative hazard
$H_0(t)=(t/\text{scale})^{\text{shape}}$, so a *constant* hazard ratio telescopes to the two-stage
Weibull-PH curve **exactly** — the joint model contains the proportional-hazards link as the
constant-trajectory special case (and the v0.33 burden link is its average-HR approximation).

In [ ]:
c = 0.5
joint_const = current_value_survival(t, np.full_like(t, c), shape=SHAPE, scale=SCALE)
two_stage   = np.exp(-(t/SCALE)**SHAPE * c)              # Weibull-PH, HR = c
print('constant HR == two-stage Weibull-PH exactly:', np.allclose(joint_const, two_stage, atol=1e-12))
baseline = current_value_survival(t, np.ones_like(t), shape=SHAPE, scale=SCALE)
print('HR == 1     == Weibull baseline exactly    :', np.allclose(baseline, np.exp(-(t/SCALE)**SHAPE), atol=1e-12))

## The payload: a time-varying (non-proportional) hazard

For a shrink-then-regrow tumor the hazard ratio is low while the tumor is small, then **rises** as
a resistant clone regrows. `ph_violation` = HR(end)/HR(8 wk) measures the departure from
proportionality — exactly 1 for a two-stage link, ≫ 1 for the resistance models.

In [ ]:
cmp = compare_joint_vs_two_stage(ds, context=ctx, alpha=1.0, t=t)
print(f"{'model':<46}{'2-stage':>8}{'joint':>7}{'PH-viol':>9}")
for r in sorted(cmp.rows, key=lambda r: -(r['joint_median'] or 0)):
    ts = 'n/r' if r['two_stage_median'] is None else f"{r['two_stage_median']:.0f}"
    jt = 'n/r' if r['joint_median'] is None else f"{r['joint_median']:.0f}"
    print(f"{r['record_id']:<46}{ts:>8}{jt:>7}{r['ph_violation']:>8.0f}x")
print()
print('rank-discordant pairs (two-stage vs joint):', cmp.rank_discordant_pairs)
print('max PH-violation                          :', f'{cmp.max_ph_violation:.0f}x')

## The ranking inverts, for a structural reason

The week-8 two-stage surrogate ranks the deep-early-shrinking two-population model *above* Claret.
The joint link, weighting the regrowth tail as a rising hazard, **inverts** it — the lighter-tail
model wins. Same trajectories, opposite ranking: two-stage-vs-joint is a model-selection axis at the
survival-link layer.

In [ ]:
claret  = joint_survival(ds, 'resistance.claret_2009.tgi', context=ctx, alpha=1.0, t=t)
two_pop = joint_survival(ds, 'resistance.nsclc_first_line.two_population', context=ctx, alpha=1.0, t=t)
print(f'week-8 two-stage : two-pop {two_pop.two_stage_median_os:.0f}  >  Claret {claret.two_stage_median_os:.0f}')
print(f'joint current-val: Claret  {claret.median_os:.0f}  >  two-pop {two_pop.median_os:.0f}   (inverted)')
print(f'two-pop HR rises 8wk {two_pop.hazard_ratio_at(8):.2f} -> end {two_pop.hazard_ratio[-1]:.0f}')

## The takeaway

The two-stage surrogate is itself a modeling choice — and a restrictive one (proportional hazards).
The joint current-value link uses the whole trajectory *dynamically*, so it represents the
non-proportional hazard a resistant regrowth induces, which every two-stage link is blind to.
Onkos ships both and quantifies the disagreement; it crowns no winner. **No individual prediction,
no therapy ranking — α is declared, never fitted, and a joint analysis never moves a tier.**